# Simple Linear Regression — Solutions
### Applied Statistical Data Analysis — Prof. Dr. Kristyna Ters | MSc Finance | FHNW

---
> ⚠️ **This file contains complete solutions. Release to students only after the submission deadline.**


In [ ]:
!pip install yfinance statsmodels --quiet

import yfinance as yf
import pandas as pd
import numpy as np
import statsmodels.api as sm
import matplotlib.pyplot as plt
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

plt.rcParams.update({
    'figure.facecolor':'white', 'axes.facecolor':'white',
    'axes.spines.top':False, 'axes.spines.right':False,
    'axes.grid':True, 'grid.alpha':0.3, 'font.size':11
})
YELLOW = '#FDE70E'; ORANGE = '#FCB310'; RED = '#C70101'
GREY   = '#4B4B4B'; BLUE = '#0E75FE'
print('✓ Libraries loaded.')

---
# Solution 1 — Dependent vs. Independent Variable

| # | y (dependent) | x (independent) | Reason |
|---|---------------|-----------------|--------|
| a | Swiss-equity portfolio return | SMI futures return | We hedge the equity exposure with futures — y depends on x |
| b | UBS return | Market return (SMI or world) | CAPM postulates stock returns are explained by market returns |
| c | Bank stock return | ΔInterest rate | We measure how rates affect bank profitability |
| d | Bond return | ΔYield | The duration relationship runs from yield changes to price changes |
| e | Future spot rate S(t+1) | Forward rate F(t) | Theory says forward predicts future spot |
| f | Mortgage rate | Treasury yield | Mortgages are priced as Treasury + credit-risk premium |

---
# Solution 2 — Manual $\hat{\beta}$

In [ ]:
x = np.array([-3, -2, -1,  1,  2,  3], dtype=float)
y = np.array([-2.5, -1.8, -0.6, 0.7, 1.9, 2.7], dtype=float)

x_bar = x.mean(); y_bar = y.mean()
num   = ((x - x_bar) * (y - y_bar)).sum()
den   = ((x - x_bar) ** 2).sum()
beta1 = num / den
beta0 = y_bar - beta1 * x_bar

print(f'x_bar = {x_bar:.4f}')
print(f'y_bar = {y_bar:.4f}')
print(f'∑(x − x_bar)(y − y_bar) = {num:.4f}')
print(f'∑(x − x_bar)²        = {den:.4f}')
print(f'β1_hat = {beta1:.4f}')
print(f'β0_hat = {beta0:.4f}')

# Verify with statsmodels
model = sm.OLS(y, sm.add_constant(x)).fit()
print(f'\nstatsmodels check: β0_hat = {model.params[0]:.4f}, β1_hat = {model.params[1]:.4f}')
print(f'Identical? {np.allclose(model.params, [beta0, beta1])}')

# Geometric check
print(f'\ny_hat at x = x_bar: {beta0 + beta1 * x_bar:.4f}   vs   y_bar = {y_bar:.4f}   ← equal')

**Answers:**
1. The hedge ratio is $\hat{\beta}_1$ ≈ 0.87 — for every 1% move in the index futures, the portfolio moves about 0.87% on average.
2. Yes, `statsmodels` returns the same numbers down to floating-point precision — the hand formula and the library implement the identical algebra.
3. By construction, $\hat{\beta}_0$ = $\bar{y}$ − $\hat{\beta}_1$·$\bar{x}$ implies $\hat{y}$ at x = $\bar{x}$ equals $\bar{y}$. The OLS line always passes through the centroid of the data.

---
# Solution 3 — Verify $\sum \hat{u} = 0$ and $\sum x \hat{u} = 0$

In [ ]:
yhat = beta0 + beta1 * x
uhat = y - yhat

tbl = pd.DataFrame({
    'x': x, 'y': y,
    'Y_hat':  yhat.round(4),
    'u_hat':  uhat.round(4),
    'u_hat²': (uhat**2).round(6),
})
print(tbl)
print('-' * 50)
print(f'Σ x        = {x.sum():.4f}')
print(f'Σ y        = {y.sum():.4f}')
print(f'Σ Y_hat        = {yhat.sum():.4f}')
print(f'Σ u_hat        = {uhat.sum():.6f}    ← essentially 0')
print(f'Σ x · u_hat    = {(x*uhat).sum():.6f}    ← essentially 0')

**Answers:**
1. The intercept $\hat{\beta}_0$ is *defined* by the first-order condition ∂($\sum \hat{u}$²)/∂β₀ = 0, which is equivalent to $\sum \hat{u} = 0$. The intercept absorbs whatever shift is needed to centre the residuals on zero.
2. Without an intercept (i.e. `sm.OLS(y, x).fit()` instead of `sm.OLS(y, sm.add_constant(x)).fit()`), the line is forced through the origin. Then $\sum \hat{u}$ is generally **not** zero, and your interpretation of residuals breaks.
3. It is a **consequence** of how OLS is computed — a mechanical algebraic identity that holds in every sample, regardless of whether the true model is linear or whether any CLRM assumption holds.

---
# Solution 4 — CAPM Beta for Nestlé

In [ ]:
STOCK = 'NESN.SW'
px = yf.download([STOCK, '^SSMI'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna() * 100

y = ret[STOCK]
X = sm.add_constant(ret['^SSMI'])
model = sm.OLS(y, X).fit()

print(f'CAPM for {STOCK} vs SMI (2019–2024)')
print(f'  β0_hat (alpha, daily %): {model.params.iloc[0]:.4f}')
print(f'  β1_hat (beta)            : {model.params.iloc[1]:.4f}')

# Scatter + fitted line
fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(ret['^SSMI'], y, color=GREY, alpha=0.5, s=15)
xx = np.linspace(ret['^SSMI'].min(), ret['^SSMI'].max(), 100)
ax.plot(xx, model.params.iloc[0] + model.params.iloc[1] * xx, color=RED, lw=2,
        label=f'β1_hat = {model.params.iloc[1]:.3f}')
ax.axhline(0, color=GREY, lw=0.5); ax.axvline(0, color=GREY, lw=0.5)
ax.set_xlabel('SMI return (%)'); ax.set_ylabel(f'{STOCK} return (%)')
ax.set_title(f'CAPM regression — {STOCK} vs SMI', fontweight='bold', loc='left')
ax.legend(loc='upper left', frameon=False)
plt.tight_layout(); plt.show()

**Answers:**
1. Nestlé's beta is typically ~0.6–0.8 — **defensive**, as expected for a consumer-staples giant whose earnings are less sensitive to the business cycle.
2. Typical ranking among Swiss blue chips: UBSG (β ≈ 1.2, cyclical bank) > Nestlé (β ≈ 0.7, defensive) ≈ Roche/Novartis (β ≈ 0.6, defensive pharma). Makes economic sense.
3. Expected loss ≈ $\hat{\beta}_1$ × −3% — for Nestlé with β ≈ 0.7, that's about −2.1%.

---
# Solution 5 — Bond Duration via Regression

In [ ]:
px_bond = yf.download('TLT', start='2019-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)['Close']
yld     = yf.download('^TNX', start='2019-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)['Close']

tlt_ret = px_bond.pct_change().dropna() * 100   # %
tnx_chg = yld.diff().dropna()                    # already in %
common  = tlt_ret.index.intersection(tnx_chg.index)
tlt_ret = tlt_ret.loc[common].squeeze()
tnx_chg = tnx_chg.loc[common].squeeze()

X = sm.add_constant(tnx_chg)
model = sm.OLS(tlt_ret, X).fit()

duration = -model.params.iloc[1]
print(f'β0_hat = {model.params.iloc[0]:.4f}')
print(f'β1_hat = {model.params.iloc[1]:.4f}')
print(f'\nImplied duration ≈ {duration:.1f} years')
print('Careful: this is NOT the fund\'s published modified duration (about 17 years).')
print('^TNX is the 10-year point while TLT holds 20y+ bonds, so the slope is duration')
print('times the sensitivity of the long end to the 10-year yield — a yield beta below')
print('one. The regression estimate is therefore biased BELOW the published figure.')

# Plots
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(tnx_chg, tlt_ret, color=GREY, alpha=0.5, s=15)
xx = np.linspace(tnx_chg.min(), tnx_chg.max(), 100)
axes[0].plot(xx, model.params.iloc[0] + model.params.iloc[1] * xx, color=RED, lw=2)
axes[0].set_xlabel('Δ 10y yield (%)'); axes[0].set_ylabel('TLT return (%)')
axes[0].set_title(f'TLT vs Δ10y yield  —  β1_hat = {model.params.iloc[1]:.2f}',
                  fontweight='bold', loc='left')
axes[1].scatter(model.fittedvalues, model.resid, color=GREY, alpha=0.5, s=15)
axes[1].axhline(0, color=RED, lw=1.5)
axes[1].set_xlabel('Fitted Y_hat'); axes[1].set_ylabel('Residual u_hat')
axes[1].set_title('Residual plot', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Answers:**
1. Read the implied duration the cell prints. The simple regression recovers a key fixed-income risk metric, but it is a *market-implied* proxy and comes out below the fund's published modified duration of about 17 years: ^TNX measures the 10-year point while TLT holds 20-year-plus bonds, so the slope is duration times the sensitivity of the long end to the 10-year yield, and that sensitivity is below one. Only under an exactly parallel shift of the whole curve would the two coincide.
2. Bond prices and yields are inversely related: when yields rise, existing fixed coupons become less attractive, prices fall. Hence the slope is negative.
3. Look for fanning over time (heteroskedasticity around 2020 and 2022 — large rate moves) and possibly mild autocorrelation. Both signal that the simple OLS standard errors will be misleading — robust standard errors (HC3 or Newey-West) would be more appropriate.

---
# Solution 6 — Mortgage Pass-Through

In [ ]:
import pandas_datareader.data as web

mort = web.DataReader('MORTGAGE30US', 'fred', '2010-01-01', '2024-12-31')
ust  = web.DataReader('DGS10',         'fred', '2010-01-01', '2024-12-31')

df = pd.concat([mort, ust], axis=1).dropna()
df.columns = ['MORT', 'UST']

X = sm.add_constant(df['UST'])
model = sm.OLS(df['MORT'], X).fit()

print(f'β0_hat (intercept) = {model.params.iloc[0]:.3f}%   ← credit-risk premium')
print(f'β1_hat (slope)     = {model.params.iloc[1]:.3f}    ← pass-through coefficient')
print(f'\nInterpretation: For every 1 percentage point rise in the 10-year Treasury yield,')
print(f'mortgage rates rise by about {model.params.iloc[1]:.2f} percentage points.')

fig, ax = plt.subplots(figsize=(12, 4.5))
ax.plot(df.index, df['MORT'], color='black', lw=1.5, label='30-yr Mortgage rate')
ax.plot(df.index, df['UST'],  color=ORANGE, lw=1.5, label='10-yr Treasury yield')
spread = df['MORT'] - df['UST']
ax.fill_between(df.index, df['UST'], df['MORT'], alpha=0.15, color=RED, label='Spread')
ax.legend(loc='upper left', frameon=False)
ax.set_ylabel('Rate (%)')
ax.set_title('Mortgage rate vs 10-yr Treasury yield, 2010–2024', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

print(f'\nAverage spread 2010–2024:  {spread.mean():.2f}%')
print(f'Spread in 2022–2023 peak:  {spread.loc["2022-09-01":"2023-09-01"].max():.2f}%')

**Answers:**
1. $\hat{\beta}_1$ is typically around 0.9–1.0 — close to perfect pass-through but slightly less than 1, suggesting banks adjust mortgage rates a bit less than one-for-one with Treasury yields in the short run.
2. The intercept $\hat{\beta}_0$ ≈ 1.5–2.0% is the credit-risk premium — what banks charge above the riskless rate to cover prepayment risk, default risk, and origination costs.
3. The spread widened sharply in 2022–2023 (post-Fed-hike cycle) to over 3% — a sign of mortgage market stress, prepayment-risk pricing changes, and reduced bank appetite for mortgage credit.

---
# Solution 7 — Heteroskedasticity in Two Periods

In [ ]:
px = yf.download(['GC=F', 'SI=F'], start='2017-01-01', end='2022-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna() * 100

calm   = ret.loc['2017-01-01':'2019-12-31']
stress = ret.loc['2020-01-01':'2022-12-31']

results = {}
for label, df in [('Calm 2017–19', calm), ('Stress 2020–22', stress)]:
    X = sm.add_constant(df['SI=F'])
    m = sm.OLS(df['GC=F'], X).fit()
    results[label] = {'β1_hat': m.params.iloc[1], 'σ(resid)': m.resid.std(), 'n': len(df)}

print(pd.DataFrame(results).T.round(4))

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for ax, (label, df) in zip(axes, [('Calm 2017–19', calm), ('Stress 2020–22', stress)]):
    X = sm.add_constant(df['SI=F'])
    m = sm.OLS(df['GC=F'], X).fit()
    ax.plot(df.index, m.resid, color=GREY, lw=0.7)
    ax.axhline(0, color=RED, lw=1)
    ax.set_title(f'{label} — σ(u_hat) = {m.resid.std():.3f}', fontweight='bold', loc='left')
    ax.set_ylabel('Residual u_hat (%)')
plt.tight_layout(); plt.show()

**Answers:**
1. The residual standard deviation is much larger in 2020–22 (COVID + war) than 2017–19. The residual scatter widens noticeably in the stressed period.
2. Assumption A2 — homoskedasticity (constant variance) — is violated. The phenomenon is called **heteroskedasticity**.
3. $\hat{\beta}_1$ is roughly similar across periods (still unbiased!), but the standard errors are wrong under naive OLS. Heteroskedasticity-robust standard errors — HC3 or White — would give correct uncertainty estimates. Formal test (Breusch-Pagan / White) comes in V5.

---
# Solution 8 — Outlier Impact

In [ ]:
px = yf.download(['SPY', 'QQQ'], start='2018-01-01', end='2019-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna() * 100

# Clean regression
X_clean = sm.add_constant(ret['QQQ'])
m_clean = sm.OLS(ret['SPY'], X_clean).fit()
beta1_clean = m_clean.params.iloc[1]

# Inject an outlier at QQQ = −10, SPY = +5
ret_inj = pd.concat([
    ret,
    pd.DataFrame({'QQQ': [-10.0], 'SPY': [5.0]},
                 index=[pd.Timestamp('2020-01-01')])
])
X_inj = sm.add_constant(ret_inj['QQQ'])
m_inj = sm.OLS(ret_inj['SPY'], X_inj).fit()
beta1_inj = m_inj.params.iloc[1]

pct_change = (beta1_inj - beta1_clean) / beta1_clean * 100
print(f'β1_hat clean       : {beta1_clean:.4f}')
print(f'β1_hat with outlier: {beta1_inj:.4f}')
print(f'Change         : {pct_change:+.1f}%')

fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(ret['QQQ'], ret['SPY'], color=GREY, alpha=0.5, s=20,
           edgecolor='black', linewidth=0.3, label='Normal days')
ax.scatter([-10], [5], color=RED, s=180, zorder=5, edgecolor='black', linewidth=0.8,
           label='Outlier (injected)')
xx = np.linspace(min(ret['QQQ'].min(), -10), ret['QQQ'].max(), 100)
ax.plot(xx, m_clean.params.iloc[0] + beta1_clean * xx,
        color='black', lw=2, label=f'Clean fit β1_hat = {beta1_clean:.2f}')
ax.plot(xx, m_inj.params.iloc[0] + beta1_inj * xx, color=RED, lw=2, ls='--',
        label=f'With-outlier fit β1_hat = {beta1_inj:.2f}')
ax.legend(loc='upper left', frameon=False)
ax.set_xlabel('QQQ return (%)'); ax.set_ylabel('SPY return (%)')
ax.set_title('Outlier impact on β1_hat', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Answers:**
1. $\hat{\beta}_1$ changes by roughly 10–25% depending on the exact data window — a single point moved the slope by tens of percent.
2. Points with extreme x-values have **high leverage** — their distance from $\bar{x}$ multiplied by the residual contributes much more to the slope than a centred point. Geometrically, the regression line pivots around ($\bar{x}$, $\bar{y}$), and far-away points exert more torque.
3. Best practice: (i) always plot first, (ii) verify the outlier is not a data error, (iii) if real, run the regression with AND without it and report both, or use robust regression methods (quantile, Huber). Dropping silently without disclosure is bad science.

---
# Solution 9 — Diagnose Residuals (example: TLT-Duration regression)

In [ ]:
# Re-run the TLT-Duration regression
px_bond = yf.download('TLT', start='2019-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)['Close']
yld     = yf.download('^TNX', start='2019-01-01', end='2024-12-31',
                       auto_adjust=True, progress=False)['Close']
tlt_ret = px_bond.pct_change().dropna() * 100
tnx_chg = yld.diff().dropna()
common  = tlt_ret.index.intersection(tnx_chg.index)
tlt_ret = tlt_ret.loc[common].squeeze()
tnx_chg = tnx_chg.loc[common].squeeze()
model = sm.OLS(tlt_ret, sm.add_constant(tnx_chg)).fit()

print(f'β0_hat = {model.params.iloc[0]:.4f}, β1_hat = {model.params.iloc[1]:.4f}\n')

# Three diagnostic plots
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# (a) residuals over time
axes[0].plot(model.resid.index, model.resid.values, color=GREY, lw=0.6)
axes[0].axhline(0, color=RED, lw=1)
axes[0].set_title('Residuals over time', fontweight='bold', loc='left')
axes[0].set_ylabel('u_hat')

# (b) residuals vs fitted
axes[1].scatter(model.fittedvalues, model.resid, color=GREY, alpha=0.5, s=15)
axes[1].axhline(0, color=RED, lw=1)
axes[1].set_title('Residuals vs fitted', fontweight='bold', loc='left')
axes[1].set_xlabel('Y_hat'); axes[1].set_ylabel('u_hat')

# (c) histogram
axes[2].hist(model.resid, bins=40, color=GREY, edgecolor='black')
axes[2].axvline(0, color=RED, lw=1)
axes[2].set_title('Histogram of residuals', fontweight='bold', loc='left')
axes[2].set_xlabel('u_hat')

plt.tight_layout(); plt.show()

**Answers:**
1. TLT-Duration regression. $\hat{\beta}_0$ ≈ 0, $\hat{\beta}_1$ ≈ −17.
2. (a) residuals over time: mild waves visible — autocorrelation possibly an issue. (b) residuals vs fitted: clear fanning around large |$\hat{Y}$| — heteroskedasticity present. (c) histogram: fat-tailed and slightly skewed — mild normality violation, but Gauss-Markov doesn't require normality so that's OK for point estimates.
3. Standard remedy for heteroskedasticity is **HC3 (or White) standard errors**: in Python, `model.fit(cov_type='HC3')`. For both heteroskedasticity AND autocorrelation: **Newey-West** with `cov_type='HAC', cov_kwds={'maxlags': 5}`.

---
# Solution 10 — Example: Apple's Tech Tilt

In [ ]:
# Hypothesis: Apple has a strong positive tech tilt — its returns load heavily on QQQ
px = yf.download(['AAPL', 'QQQ'], start='2019-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
ret = px.pct_change().dropna() * 100

y = ret['AAPL']
X = sm.add_constant(ret['QQQ'])
model = sm.OLS(y, X).fit()

print(f'AAPL on QQQ regression:')
print(f'  β0_hat (alpha): {model.params.iloc[0]:.4f}%')
print(f'  β1_hat (beta) : {model.params.iloc[1]:.4f}')

fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))
axes[0].scatter(ret['QQQ'], y, color=GREY, alpha=0.5, s=15)
xx = np.linspace(ret['QQQ'].min(), ret['QQQ'].max(), 100)
axes[0].plot(xx, model.params.iloc[0] + model.params.iloc[1] * xx, color=RED, lw=2,
             label=f'β1_hat = {model.params.iloc[1]:.2f}')
axes[0].set_xlabel('QQQ return (%)'); axes[0].set_ylabel('AAPL return (%)')
axes[0].set_title('AAPL vs QQQ', fontweight='bold', loc='left')
axes[0].legend(loc='upper left', frameon=False)
axes[1].scatter(model.fittedvalues, model.resid, color=GREY, alpha=0.5, s=15)
axes[1].axhline(0, color=RED, lw=1.5)
axes[1].set_xlabel('Fitted Y_hat'); axes[1].set_ylabel('Residual u_hat')
axes[1].set_title('Residual plot', fontweight='bold', loc='left')
plt.tight_layout(); plt.show()

**Narrative:** Apple's $\hat{\beta}_1$ versus QQQ is typically around 1.15 — meaningfully greater than 1, confirming Apple moves a bit more than the broader tech index. The intercept $\hat{\beta}_0$ is small and not economically meaningful (we'll test its statistical significance in V5). The residual plot shows mild fanning around extreme fitted values, suggesting heteroskedasticity during crisis periods — robust standard errors would be appropriate for any inference.

---
# 🔥 Challenge — Forward Rate Puzzle (EUR/USD)

In [ ]:
# Simplified test: use one-month-lagged spot as a forward proxy
px = yf.download('EURUSD=X', start='2010-01-01', end='2024-12-31',
                 auto_adjust=True, progress=False)['Close']
px = px.dropna()
monthly = px.resample('ME').last()

F_t   = monthly.shift(1).dropna()             # 'forward' (lagged)
S_tp1 = monthly.loc[F_t.index]                 # future spot
S_tp1 = pd.Series(S_tp1.squeeze(), index=F_t.index)
F_t   = pd.Series(F_t.squeeze(),   index=F_t.index)

X = sm.add_constant(F_t)
model = sm.OLS(S_tp1, X).fit()

print(f'EUR/USD LEVEL regression (monthly, 2010–2024) — a deliberate counter-example:')
print(f'  β0_hat = {model.params.iloc[0]:.4f}')
print(f'  β1_hat = {model.params.iloc[1]:.4f}')
print(f'\nRead this the right way round. β1_hat lands on 1 MECHANICALLY, and that is the')
print('lesson, not the test. Substituting the lagged spot for the forward turns the')
print('regression into S_t+1 on S_t, i.e. a pure random walk, and both sides carry a')
print('unit root, so the slope is pinned at 1 whatever the forward market is doing.')
print('\nThe real forward premium puzzle is a regression on CHANGES:')
print('    Δs_t+1 = β0 + β1 (f_t − s_t) + u_t,   unbiasedness: β1 = 1.')
print('Estimated on genuine forwards (or the interest-rate differential from FRED) the')
print('slope comes out far below 1 and is frequently negative (Fama, 1984). We cannot')
print('reproduce it here because yfinance carries no forward quotes.')

---
*Applied Statistical Data Analysis | Prof. Dr. Kristyna Ters | FHNW School of Business | HS 2026*